# Közel inkompresszibilis Q4 elem — LowLevelFEM.jl

A BME-s TDK-dolgozat **8×2 elemes konzol-benchmarkja**: teljes integrálás (Q4-FI) és szelektív redukált integrálás (Q4-SRI). A bal él befogott, a jobb élen 1000 N függőleges erő hat. Síkalakváltozás, $E=200$ GPa, vastagság 0,1 m.

**Nincs új elemtípus, nincs saját elemmátrix-összeállítás, nincs core-módosítás.** A Q4-SRI elmozdulásai a dolgozat szerinti kondenzált Q4-UP0 elemmel ekvivalensek.

In [3]:
using LowLevelFEM, Statistics

# Háló, elmozdulásmező, peremfeltételek és terhelés
structured_rect_mesh(lx=8, ly=1, nx=8, ny=2, order=1)
U = Field([Material("body")], type=:VectorField, dim=2, fieldName=:u)
bc = [BoundaryCondition("left", field=U, ux=0, uy=0)]
f = ∫(U ⋅ [0.0, -1e4]; Γ="right")  # 1000 N összerő, 0,1 m vastagságnál

tags, xyz, _ = gmsh.model.mesh.getNodes()
right = tags[findall(abs.(xyz[1:3:end] .- 8) .< 1e-9)]

3-element Vector{UInt64}:
 0x0000000000000002
 0x0000000000000003
 0x000000000000000c

In [4]:
# Poisson-tényező szerinti vizsgálat
E = 200e9
νs = [0.30, 0.45, 0.49, 0.499, 0.4999]
R = zeros(length(νs), 3)

for (i, ν) in enumerate(νs)
    G = E / (2(1 + ν))
    κ = E / (3(1 - 2ν))
    Ddev = G * [4/3 -2/3 0; -2/3 4/3 0; 0 0 1]

    # Deviatorikus rész: teljes integrálás mindkét módszernél
    Kdev = ∫(SymGrad(U) ⋅ Ddev ⋅ SymGrad(U))

    # Térfogati rész: teljes (FI), illetve redukált (SRI) integrálás
    Kfi  = Kdev + ∫(Div(U) ⋅ κ ⋅ Div(U))
    Ksri = Kdev + ∫(Div(U) ⋅ κ ⋅ Div(U); gauss=:reduced)

    ufi  = solveField(Kfi,  f; support=bc)
    usri = solveField(Ksri, f; support=bc)
    R[i, :] = [ν, mean(ufi.a[2 .* right, 1]), mean(usri.a[2 .* right, 1])]
end

# Oszlopok: ν, Q4-FI, Q4-SRI; normalizálva az SRI ν=0,30 értékére
round.(hcat(R[:,1], R[:,2] ./ R[1,3], R[:,3] ./ R[1,3]); digits=5)

5×3 Matrix{Float64}:
 0.3     0.86445  1.0
 0.45    0.57225  0.93186
 0.49    0.22431  0.89628
 0.499   0.04199  0.88687
 0.4999  0.01837  0.8859

**Várható eredmény:** az eredeti, lefuttatott notebook alapján $\nu=0{,}4999$ esetén a normalizált értékek **Q4-FI: 0,01837**, **Q4-SRI: 0,88590**. A BME-dolgozat 7.1. táblázata kerekítve ugyanezeket az értékeket közli: **0,018** és **0,886**.

Az új, rövidített notebook futtatása külön ellenőrzendő.